# مرشّح الترددات العالية (High-Pass Filter)

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 7

---

## What this notebook does

A high-pass filter removes **low-frequency components** (below 1 Hz) from the EEG signal. This eliminates:

- DC offset (constant voltage shift)
- Baseline drift (slow upward/downward trends)
- Low-frequency artifacts (sweat, breathing, electrode movement)

## What you should expect to see

After filtering, the signal should look **flatter** on the Y-axis (the slow drift disappears), but it will still be **noisy** because high-frequency noise is not removed yet. We will handle that in the low-pass filter notebook.

## Key parameters

| Parameter | Value | Meaning |
|-----------|-------|---------|
| Cutoff | 1 Hz | Frequencies below 1 Hz are removed |
| Order | 4 | Steepness of the filter roll-off |
| Method | Butterworth | Flat response in the passband |
| filtfilt | Yes | Zero-phase (no time delay) |

## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. Clone the resources repo and download one subject

We download only **Subject 72 recordings, about 8 MB total). The full dataset has 20 subjects.

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 7

## 3. Load the EEG signal

We load subject 7, experiment 1, session 2. The signal has 4 channels (P4, Cz, F8, T7) recorded at 200 Hz. We will work with the **P4 channel** (parietal region) for this example.

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=7, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')
print(f'Sampling rate: {fs} Hz')

## 4. Apply the high-pass filter

We use a **4th-order Butterworth high-pass filter** with a cutoff of 1 Hz. The `filtfilt` function applies the filter twice (forward then backward), which eliminates any phase delay. This is important for EEG analysis where timing matters.

In [ ]:
from scipy import signal

def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs  # Nyquist frequency = 100 Hz
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False)
    filtered = signal.filtfilt(b, a, data)
    return filtered

filtered_hp = butter_highpass_filter(channel_data, cutoff=1.0, fs=fs)
print(f'Filter applied: high-pass at {1.0} Hz, order {4}')
print(f'Nyquist frequency: {0.5*fs} Hz')

## 5. Interactive plot: before vs after

The plot below is **interactive**. You can:
- **Zoom** by clicking and dragging
- **Pan** by clicking the pan tool
- **Hover** to see exact values

**What to look for:**
- The raw signal (top) has a slow drift, visible as a gradual upward or downward trend
- The filtered signal (bottom) is centered around zero, the drift is gone
- The filtered signal is still noisy (high-frequency oscillations) — this is expected, the low-pass filter will remove that

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0  # convert ms to seconds

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Raw EEG (P4)', 'After high-pass filter (1 Hz)'))

fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot],
                         name='Raw', line=dict(color='gray', width=0.5)),
               row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_hp[:n_plot],
                         name='Filtered', line=dict(color='green', width=0.5)),
               row=2, col=1)

fig.update_layout(height=600, title_text='High-Pass Filter: Before vs After',
                  xaxis2_title='Time (s)', yaxis_title='EEG (uV)',
                  yaxis2_title='EEG (uV)')
fig.show()

## 6. What did we learn?

- The high-pass filter at 1 Hz **removed the DC offset and slow drift** from the signal
- The signal is now centered around zero on the Y-axis
- **High-frequency noise is still present** — the signal looks jagged
- The next step is to apply a **low-pass filter** to remove that noise (see the next notebook)